<a href="https://colab.research.google.com/github/kangwonlee/nmisp/blob/main/60_linear_algebra_2/205_Google_PageRank.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


In [ ]:
# This cell is for the Google Colaboratory
# https://stackoverflow.com/a/63519730
if 'google.colab' in str(get_ipython()):
  path_py = '/content/nmisp_py'

  import os
  if not os.path.exists(path_py):
    import subprocess
    subprocess.run(
        ('git', 'clone', 'https://github.com/kwlee2025cpp/nmisp_py')
    )
  assert os.path.exists(path_py)

  import sys
  sys.path.insert(0, path_py)


In [ ]:
# 그래프, 수학 기능 추가
# Add graph and math features
import matplotlib.pyplot as plt
import numpy as np
import numpy.linalg as nl



# 구글 페이지랭크 알고리듬<br>The PageRank Algorithm of Google



* 위키백과 기여자, '페이지랭크', 위키백과, , 2017년 9월 1일, 02:44 UTC, <https://ko.wikipedia.org/w/index.php?title=%ED%8E%98%EC%9D%B4%EC%A7%80%EB%9E%AD%ED%81%AC&oldid=19481213> [2018년 7월 31일에 접근]
* Wikipedia contributors, 'PageRank', Wikipedia, The Free Encyclopedia, 16 July 2018, 19:43 UTC, <https://en.wikipedia.org/w/index.php?title=PageRank&oldid=850584505> [accessed 31 July 2018]



다음 그림을 살펴 보자.<br>
Let's take a look a the following figure.



![pagerank](https://user-images.githubusercontent.com/17876446/246726501-117c56e0-5b0c-4bc4-a84f-27d706f82aa6.jpg)



위 연결 상태를 다음과 같은 행렬로 표시할 수 있다.<br>The connectivity status can be represented as a matrix as follows.



| m | A  | B  | C  | D  | E  | F  | G1 | G2 | G3 | G4 | G5 |
|:---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| A   | 0  | 0  | 0  | 1  | 0  | 0  | 0  |  0 |  0 |  0 |  0 |
| B   | 0  | 0  | 1  | 1  | 1  | 1  | 1  |  1 |  1 |  0 |  0 |
| C   | 0  | 1  | 0  | 0  | 0  | 0  | 0  |  0 |  0 |  0 |  0 |
| D   | 0  | 0  | 0  | 0  | 1  | 0  | 0  |  0 |  0 |  0 |  0 |
| E   | 0  | 0  | 0  | 0  | 0  | 1  | 1  |  1 |  1 |  1 |  1 |
| F   | 0  | 0  | 0  | 0  | 1  | 0  | 0  |  0 |  0 |  0 |  0 |
| G1  | 0  | 0  | 0  | 0  | 0  | 0  | 0  |  0 |  0 |  0 |  0 |
| G2  | 0  | 0  | 0  | 0  | 0  | 0  | 0  |  0 |  0 |  0 |  0 |
| G3  | 0  | 0  | 0  | 0  | 0  | 0  | 0  |  0 |  0 |  0 |  0 |
| G4  | 0  | 0  | 0  | 0  | 0  | 0  | 0  |  0 |  0 |  0 |  0 |
| G5  | 0  | 0  | 0  | 0  | 0  | 0  | 0  |  0 |  0 |  0 |  0 |



예를 들어 첫 행에서 A 절점으로 들어오는 연결선은 D 절점에서 오는 것 하나 뿐이다.<br>
For example, in the first row, node A has only one incoming link from node D.



In [ ]:
m = np.array(
    [
        [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0],
        [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    ], dtype=float
)



각 열의 합이 1이 되도록 만든다.<br>Make sum of each column to one.



In [ ]:
s = m.sum(axis=0)



In [ ]:
for j in range(m.shape[1]):
    if 1 < s[j]:
        si = 1.0 / s[j]
        for i in range(m.shape[0]):
            m[i, j] *= si



In [ ]:
m



### 전이 행렬을 Hinton 다이어그램으로<br>The transition matrix as a Hinton diagram

실제 링크 행렬은 대부분 0 인 *희소행렬* 이다. Hinton 다이어그램에서는 0 인 원소가 말 그대로 빈 칸으로 보여, 색상지도(`matshow`)보다 희소성이 훨씬 잘 드러난다.<br>
Real link matrices are mostly zero (sparse). In a Hinton diagram a zero entry is literally empty space, so the sparsity is far clearer than under a colormap, where a near-zero entry is indistinguishable from any other small value.


In [ ]:
import matshow

fig, ax = plt.subplots()
matshow.hinton(np.array(m), ax=ax)
ax.set_title('PageRank transition matrix (Hinton)')
plt.show()


이제 행렬의 고유벡터를 구한다.<br>Now let's find the eigenvector of the matrix.



In [ ]:
eval, evac = nl.eig(m)



In [ ]:
eval



In [ ]:
plt.matshow(evac)
plt.colorbar()
plt.axis('equal')



In [ ]:
evac[:, 0]


### 거듭제곱 반복으로 페이지랭크 구하기<br>Finding PageRank by power iteration

실제 웹에서는 행렬이 너무 커서 `nl.eig` 로 한 번에 고유벡터를 구할 수 없다. 대신 *거듭제곱 반복* 으로 지배적 고유벡터를 점진적으로 찾는다.<br>
On the real web the matrix is far too large to diagonalize with `nl.eig`; instead the dominant eigenvector is found incrementally by *power iteration*.

여기에는 두 가지 보정이 필요하다: ① 막다른 노드(나가는 링크가 없는 페이지)는 모든 페이지로 균등하게 연결된 것으로 보고, ② *감쇠 계수* `d`(보통 0.85)로 가끔 임의의 페이지로 점프(텔레포트)하게 한다. 이 둘이 *구글 행렬* 을 기약·비주기 행렬로 만들어 유일한 정상분포로의 수렴을 보장한다.<br>
Two corrections are needed: (1) a *dangling node* (a page with no out-links) is treated as linking uniformly to every page, and (2) a *damping factor* `d` (typically 0.85) lets the surfer occasionally teleport to a random page. Together they make the *Google matrix* irreducible and aperiodic, guaranteeing convergence to a unique stationary distribution.


In [ ]:
n_page = m.shape[0]

# ① 막다른 노드 처리 / handle dangling nodes (columns that sum to zero)
dangling = (m.sum(axis=0) == 0)
m_stochastic = m.astype(float).copy()
m_stochastic[:, dangling] = 1.0 / n_page

# ② 감쇠 + 텔레포트로 구글 행렬 구성 / build the Google matrix
d = 0.85
google = d * m_stochastic + (1.0 - d) / n_page

# 균등 분포에서 시작해 거듭제곱 반복 / power-iterate from a uniform start
rank = np.ones(n_page) / n_page
rank_history = [rank.copy()]
for _ in range(40):
    rank = google @ rank
    rank_history.append(rank.copy())

rank_columns = [r.reshape(-1, 1) for r in rank_history]
matshow.hinton_step_slider(rank_columns, description='iter')


각 페이지 순위가 반복에 따라 수렴한다. 마지막으로 구글 행렬의 지배적 고유벡터(`nl.eig`)와 일치함을 확인한다.<br>
Each page's rank converges over the iterations; finally we confirm it matches the dominant eigenvector of the Google matrix from `nl.eig`.


In [ ]:
ax = matshow.element_trace(
    rank_columns,
    indices=[(i, 0) for i in range(n_page)],
    labels=[f'p{i}' for i in range(n_page)],
)
ax.set_title('page ranks converging over power iterations')
plt.show()


In [ ]:
# 거듭제곱 반복 결과와 구글 행렬의 지배적 고유벡터 비교
# compare with the Google matrix's dominant eigenvector
eval_g, evac_g = nl.eig(google)
idx = np.argmin(np.abs(eval_g - 1.0))
dominant = np.abs(evac_g[:, idx].real)
dominant = dominant / dominant.sum()

print('power iteration :', np.round(rank_history[-1], 4))
print('dominant eigvec :', np.round(dominant, 4))


## Final Bell<br>마지막 종



In [ ]:
# stackoverfow.com/a/24634221
import os
os.system("printf '\a'");

